# Decision Tree Root Node Split Implementation
This notebook implements the logic to determine the best split for the root node of a Decision Tree using the Pima Indians Diabetes Dataset.

In [ ]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv('diabetes.csv')
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

print(f"Dataset loaded with {df.shape[0]} samples and {df.shape[1]-1} features.")

Dataset loaded with 768 samples and 8 features.


### Implementation of Entropy and Gini Index Functions

In [ ]:
def entropy(y):
    if len(y) == 0: return 0
    p = np.mean(y)
    if p == 0 or p == 1: return 0
    return - (p * np.log2(p) + (1 - p) * np.log2(1 - p))

def gini(y):
    if len(y) == 0: return 0
    p = np.mean(y)
    return 1 - (p**2 + (1 - p)**2)

def calculate_gain(y_parent, y_left, y_right, criterion='entropy'):
    w_l = len(y_left) / len(y_parent)
    w_r = len(y_right) / len(y_parent)
    if criterion == 'entropy':
        return entropy(y_parent) - (w_l * entropy(y_left) + w_r * entropy(y_right))
    else:
        return gini(y_parent) - (w_l * gini(y_left) + w_r * gini(y_right))

### Logic to Determine Best Split
We iterate through every feature and every possible threshold (midpoints of sorted unique values) to find the maximum Information Gain.

In [ ]:
features = X.columns
best_overall_split = {'gain': -1}
best_overall_gini = {'gain': -1}

for col in features:
    vals = X[col].values
    unique_vals = np.sort(np.unique(vals))
    thresholds = (unique_vals[:-1] + unique_vals[1:]) / 2

    best_feat_e = -1
    best_thresh_e = None
    best_feat_g = -1
    best_thresh_g = None

    for t in thresholds:
        mask = vals <= t
        y_l, y_r = y[mask], y[~mask]

        e_gain = calculate_gain(y, y_l, y_r, 'entropy')
        g_gain = calculate_gain(y, y_l, y_r, 'gini')

        if e_gain > best_feat_e:
            best_feat_e, best_thresh_e = e_gain, t
        if g_gain > best_feat_g:
            best_feat_g, best_thresh_g = g_gain, t

    print(f"Feature: {col:25} | Best Thresh (Entropy): {best_thresh_e:6.2f} | Gain: {best_feat_e:.4f}")
    print(f"Feature: {'':25} | Best Thresh (Gini):    {best_thresh_g:6.2f} | Gain: {best_feat_g:.4f}")

    if best_feat_e > best_overall_split['gain']:
        best_overall_split = {'feature': col, 'thresh': best_thresh_e, 'gain': best_feat_e}
    if best_feat_g > best_overall_gini['gain']:
        best_overall_gini = {'feature': col, 'thresh': best_thresh_g, 'gain': best_feat_g}

print("\n" + "="*50)
print(f"FINAL BEST SPLIT (ENTROPY): {best_overall_split['feature']} at {best_overall_split['thresh']}")
print(f"FINAL BEST SPLIT (GINI):    {best_overall_gini['feature']} at {best_overall_gini['thresh']}")

Feature: Pregnancies               | Best Thresh (Entropy):   6.50 | Gain: 0.0392
Feature:                           | Best Thresh (Gini):      6.50 | Gain: 0.0256
Feature: Glucose                   | Best Thresh (Entropy): 127.50 | Gain: 0.1308
Feature:                           | Best Thresh (Gini):    127.50 | Gain: 0.0825
Feature: BloodPressure             | Best Thresh (Entropy):  69.00 | Gain: 0.0140
Feature:                           | Best Thresh (Gini):     69.00 | Gain: 0.0087
Feature: SkinThickness             | Best Thresh (Entropy):  31.50 | Gain: 0.0169
Feature:                           | Best Thresh (Gini):     31.50 | Gain: 0.0109
Feature: Insulin                   | Best Thresh (Entropy): 121.00 | Gain: 0.0268
Feature:                           | Best Thresh (Gini):    121.00 | Gain: 0.0174
Feature: BMI                       | Best Thresh (Entropy):  27.85 | Gain: 0.0749
Feature:                           | Best Thresh (Gini):     29.85 | Gain: 0.0429
Feature: Diabete

### Results and Documentation

1. **Feature Search**: The algorithm successfully checks all 8 numerical features.
2. **Threshold Calculation**:  For each feature, sort the unique values and calculate the midpoints to ensure that we test all mathematically relevant split points
3. **Comparison**:
   - The best split based on Entropy and Gini Index for the root node is typically the same (Glucose is usually the most predictive feature for this dataset).
   - **Gini Index** focuses on minimizing misclassification impurity.
   - **Entropy** focuses on maximizing Information Gain based on the bits of information required to identify the class.